# Isolation Forest Base Model: Kiva Loan Transparency
**ICS 3202 Project Deliverable 2** | Group ML engine: flag fund allocations that look potentially misallocated.

**Approach:** Kiva has no fraud label, so this is *unsupervised anomaly detection*. Isolation Forest isolates unusual loans in fewer random splits. Records that are easy to isolate get flagged.

In [ ]:
from google.colab import userdata
import os
os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/access_token','w') as f:
    f.write(userdata.get('KAGGLE_KEY'))
os.chmod('/root/.kaggle/access_token', 0o600)
!pip install -q kaggle
!kaggle datasets download -d kiva/data-science-for-good-kiva-crowdfunding
!unzip -q -o data-science-for-good-kiva-crowdfunding.zip

## 1. Load data

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
df = pd.read_csv('kiva_loans.csv')
print(df.shape)
df.head()

## 2. Data cleaning (raw data to modelling dataset)
Steps: check missing values, remove duplicates, fix date types, drop rows missing the numeric fields the model needs, remove invalid loan amounts, and tidy text.

In [ ]:
missing = df.isnull().sum()
print(missing[missing > 0])

In [ ]:
clean = df.copy()

# 1. Duplicates
print('Duplicate loan ids:', clean.duplicated(subset='id').sum())
clean = clean.drop_duplicates(subset='id')

# 2. Convert date columns from text to datetime
for c in ['posted_time','disbursed_time','funded_time','date']:
    clean[c] = pd.to_datetime(clean[c], errors='coerce', utc=True)

# 3. Drop rows missing the numeric fields needed by the model
before = len(clean)
clean = clean.dropna(subset=['funded_amount','loan_amount','term_in_months','lender_count'])
print('Rows dropped (missing numeric fields):', before - len(clean))

# 4. Remove invalid loan amounts
before = len(clean)
clean = clean[clean['loan_amount'] > 0]
print('Rows dropped (loan_amount <= 0):', before - len(clean))

# 5. Text columns: fill missing and tidy
clean['use'] = clean['use'].fillna('unknown').str.strip()
clean['sector'] = clean['sector'].str.strip()
clean['activity'] = clean['activity'].str.strip()

print('Final shape:', clean.shape)

## 3. Feature engineering

In [ ]:
clean['funded_ratio'] = clean['funded_amount'] / clean['loan_amount']
clean['funding_gap'] = clean['loan_amount'] - clean['funded_amount']
clean['days_to_fund'] = (clean['funded_time'] - clean['posted_time']).dt.total_seconds() / 86400
clean['days_to_fund'] = clean['days_to_fund'].fillna(clean['days_to_fund'].median())

features = ['loan_amount','funded_ratio','term_in_months','lender_count','days_to_fund']
clean[features].describe()

## 4. Exploratory data analysis (4+ observations)

In [ ]:
# Obs 1: loan amount distribution
print(clean['loan_amount'].describe())
clean['loan_amount'].clip(upper=clean['loan_amount'].quantile(0.99)).hist(bins=50)
plt.title('Loan amount (clipped at 99th percentile)'); plt.show()

**Observation 1:** *(fill in from output: e.g. the median loan is X but the max is Y, so the distribution is heavily right-skewed with large outliers.)*

In [ ]:
# Obs 2: how many loans are fully funded?
print('Share fully funded:', (clean['funded_ratio'] >= 1).mean().round(3))
print('Share funded below 90%:', (clean['funded_ratio'] < 0.9).mean().round(3))
clean['funded_ratio'].hist(bins=50); plt.title('Funded ratio'); plt.show()

**Observation 2:** *(fill in: what share is fully funded, and what share is under-funded, which is our main transparency signal.)*

In [ ]:
# Obs 3: sector mix
clean['sector'].value_counts().plot(kind='barh'); plt.title('Loans per sector'); plt.show()

**Observation 3:** *(fill in: which sectors dominate the data.)*

In [ ]:
# Obs 4: relationship between loan size and number of lenders
print(clean[['loan_amount','lender_count','term_in_months','days_to_fund']].corr().round(2))

**Observation 4:** *(fill in: e.g. loan_amount and lender_count are strongly/weakly correlated, so unusually large loans with few lenders are suspicious.)*

## 5. Base model: Isolation Forest
Features are scaled first. The model returns -1 for anomalies and 1 for normal records.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest

X = StandardScaler().fit_transform(clean[features])

iso = IsolationForest(n_estimators=100, contamination=0.05, random_state=42, n_jobs=-1)
clean['iso_flag'] = (iso.fit_predict(X) == -1).astype(int)
clean['anomaly_score'] = -iso.score_samples(X)   # higher = more anomalous
print('Flagged loans:', clean['iso_flag'].sum(), f"({clean['iso_flag'].mean():.1%})")

## 6. Validating the base model
There are no true labels, so we check validity in three ways:
1. Tune `contamination` and see how results change.
2. Compare flags against a simple rule-based flag (funded below 90% of the loan amount).
3. Inspect flagged loans by hand.

*Note: agreement with the rule is a sanity check, not an accuracy score, because `funded_ratio` is also a model input.*

In [ ]:
from sklearn.metrics import precision_score, recall_score

clean['rule_flag'] = (clean['funded_ratio'] < 0.9).astype(int)

rows = []
for c in [0.01, 0.03, 0.05, 0.10]:
    m = IsolationForest(n_estimators=100, contamination=c, random_state=42, n_jobs=-1)
    f = (m.fit_predict(X) == -1).astype(int)
    rows.append({'contamination': c, 'flagged': f.sum(),
                 'precision_vs_rule': precision_score(clean['rule_flag'], f),
                 'recall_vs_rule': recall_score(clean['rule_flag'], f)})
pd.DataFrame(rows).round(3)

In [ ]:
# Inspect the most anomalous loans
clean.sort_values('anomaly_score', ascending=False)[
    ['id','sector','activity','country','loan_amount','funded_amount','funded_ratio',
     'lender_count','term_in_months','anomaly_score']].head(10)

## 7. Further validation (planned for the final stage)
To check the base model, we will build other models that follow the same process (same cleaned features, same scaling) and compare their results with Isolation Forest:
- **Local Outlier Factor (LOF)**: density-based anomaly detection.
- **K-means (elbow method)**: loans far from their cluster centre are treated as anomalies.
- **Use vs sector/activity mismatch**: text-based check for loans whose stated `use` does not match their declared `sector`.

Comparison criteria: overlap between flagged sets, agreement with the rule-based flag, and manual review of flagged loans.

In [ ]:
# Quick preview of the comparison (sampled, since LOF is slow on 600k+ rows)
from sklearn.neighbors import LocalOutlierFactor
samp = clean.sample(20000, random_state=42)
Xs = StandardScaler().fit_transform(samp[features])
lof_flag = (LocalOutlierFactor(n_neighbors=20, contamination=0.05).fit_predict(Xs) == -1).astype(int)
iso_s = (IsolationForest(contamination=0.05, random_state=42).fit_predict(Xs) == -1).astype(int)
print('Overlap of LOF and Isolation Forest flags:', ((lof_flag == 1) & (iso_s == 1)).sum(), 'of', iso_s.sum())